<a href="https://colab.research.google.com/github/muraleee/collab-stuff/blob/main/RAG_KK_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain-openai langchain-core langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 4.3 MB/s eta 0:00:00


In [5]:
"""
Agentic Chunking Demo
Using LLM to intelligently split documents based on semantic meaning

This script demonstrates **Agentic Chunking** - the most advanced chunking method
where an AI model analyzes the document and decides optimal split points based on
topic shifts and semantic coherence, rather than arbitrary character counts.
"""
import os
import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google.colab import userdata

print("🤖 Agentic Chunking Demo")
print("=" * 50)

API_KEY = userdata.get('OPENAI_API_KEY')

API_BASE = os.environ.get("OPENAI_API_BASE", "https://api.openai.com/v1")
MODEL_NAME = "gpt-5-nano"

if not API_KEY:
    print("❌ Error: OPENAI_API_KEY not found.")
    print("Please ensure the environment is configured correctly.")
    sys.exit(1)

print(f"🔌 API Endpoint: {API_BASE}")
print(f"🧠 Model: {MODEL_NAME}")
print()


🤖 Agentic Chunking Demo
🔌 API Endpoint: https://api.openai.com/v1
🧠 Model: gpt-5-nano



In [3]:
import os
import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google.colab import userdata



# Sample document with multiple distinct topics
sample_document = """
TechCorp Company Overview

Company History: Founded in 1995 in a garage in Silicon Valley, TechCorp started as a small software consultancy. By 2000, it had grown to 500 employees and went public.
The early years were marked by rapid expansion and the release of its flagship product, the TechOS. The founders, Jane Smith and John Doe, built the company on principles of innovation and customer focus.

Product Lineup: Today, TechCorp offers a wide range of enterprise software solutions. The CloudSuite is our most popular offering, providing scalable cloud infrastructure for businesses of all sizes.
We also offer DataGuard for enterprise security, protecting sensitive data with military-grade encryption. AI-Core handles machine learning integration, making AI accessible to non-technical teams. Each product is designed to work seamlessly with the others.

Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. Remote work must be conducted using company-approved devices with VPN access enabled.
All employees must be available during core hours (10 AM - 4 PM) and maintain regular communication with their team. Remote work is not a substitute for childcare or eldercare.

Future Vision: Looking ahead, TechCorp is betting big on quantum computing. We plan to invest $1B over the next 5 years in R&D for quantum technologies. Our goal is to be the first company to offer
commercial quantum cloud services by 2030. This investment will create new positions for quantum researchers and engineers across all our locations.
"""

print("📄 Sample Document:")
print(f"Length: {len(sample_document)} characters")
print(f"Contains 4 distinct topics: History, Products, Remote Work, Future")
print()

# First, let's compare with basic chunking
print("🔧 Comparison: Basic Chunking vs Agentic Chunking")
print("-" * 50)

# Basic character-based chunking
basic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

basic_chunks = basic_splitter.split_text(sample_document)
print(f"\n📊 Basic Chunking Result: {len(basic_chunks)} chunks")
print("   (Based on character count, may split mid-topic)")
for i, chunk in enumerate(basic_chunks, 1):
    preview = chunk[:60].replace('\n', ' ').strip()
    print(f"   Chunk {i}: {preview}...")
print()

📄 Sample Document:
Length: 1568 characters
Contains 4 distinct topics: History, Products, Remote Work, Future

🔧 Comparison: Basic Chunking vs Agentic Chunking
--------------------------------------------------

📊 Basic Chunking Result: 6 chunks
   (Based on character count, may split mid-topic)
   Chunk 1: TechCorp Company Overview...
   Chunk 2: Company History: Founded in 1995 in a garage in Silicon Vall...
   Chunk 3: Product Lineup: Today, TechCorp offers a wide range of enter...
   Chunk 4: making AI accessible to non-technical teams. Each product is...
   Chunk 5: Remote Work Policy: Employees may work remotely up to 3 days...
   Chunk 6: Future Vision: Looking ahead, TechCorp is betting big on qua...



In [6]:
# Agentic chunking using LLM
def agentic_chunking(text):
    """
    Uses an LLM to split text into semantically distinct chunks.
    The AI analyzes topic shifts and creates meaningful boundaries.
    """
    print("🤔 Agent is analyzing the document for semantic topic shifts...")

    llm = ChatOpenAI(
        model=MODEL_NAME,
        openai_api_key=API_KEY,
        openai_api_base=API_BASE,
        temperature=0  # Deterministic output for consistency
    )

    # The Prompt: Instruct the LLM to act as a "Chunking Agent"
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert document editor specializing in semantic document analysis.
Your task is to split the provided text into semantically distinct chunks based on topic shifts.

Rules:
1. Keep related sentences together - don't break up a single topic
2. Split ONLY when the topic changes significantly (e.g., History -> Products -> Policy -> Future)
3. Each chunk should be about ONE coherent topic
4. Output the chunks separated by '---SPLIT---'
5. Do not modify the original text - just split it at appropriate boundaries
6. Include section headers with their content in the same chunk"""),
        ("user", "{text}")
    ])

    chain = prompt | llm | StrOutputParser()

    try:
        response = chain.invoke({"text": text})
        # Split the response by our delimiter and clean up
        chunks = [c.strip() for c in response.split("---SPLIT---") if c.strip()]
        return chunks
    except Exception as e:
        print(f"\n❌ API Error: {e}")
        return []

# Run agentic chunking
agentic_chunks = agentic_chunking(sample_document)

if agentic_chunks:
    print(f"\n📊 Agentic Chunking Result: {len(agentic_chunks)} chunks")
    print("   (Based on semantic meaning and topic shifts)")
    print()

    for i, chunk in enumerate(agentic_chunks, 1):
        # Identify the likely topic from the chunk
        if "History" in chunk or "Founded" in chunk:
            topic = "Company History"
        elif "Product" in chunk or "CloudSuite" in chunk:
            topic = "Products"
        elif "Remote" in chunk or "work" in chunk.lower():
            topic = "Remote Work Policy"
        elif "Future" in chunk or "quantum" in chunk.lower():
            topic = "Future Vision"
        else:
            topic = "General"

        print(f"📦 Chunk {i} - Topic: {topic}")
        print(f"   Length: {len(chunk)} characters")
        preview = chunk[:80].replace('\n', ' ').strip()
        print(f"   Preview: {preview}...")
        print()

🤔 Agent is analyzing the document for semantic topic shifts...

📊 Agentic Chunking Result: 4 chunks
   (Based on semantic meaning and topic shifts)

📦 Chunk 1 - Topic: Company History
   Length: 401 characters
   Preview: TechCorp Company Overview  Company History: Founded in 1995 in a garage in Silic...

📦 Chunk 2 - Topic: Products
   Length: 458 characters
   Preview: Product Lineup: Today, TechCorp offers a wide range of enterprise software solut...

📦 Chunk 3 - Topic: Remote Work Policy
   Length: 355 characters
   Preview: Remote Work Policy: Employees may work remotely up to 3 days per week with manag...

📦 Chunk 4 - Topic: Future Vision
   Length: 346 characters
   Preview: Future Vision: Looking ahead, TechCorp is betting big on quantum computing. We p...



In [ ]:
"""
Agentic Chunking Demo
Using LLM to intelligently split documents based on semantic meaning

This script demonstrates **Agentic Chunking** - the most advanced chunking method
where an AI model analyzes the document and decides optimal split points based on
topic shifts and semantic coherence, rather than arbitrary character counts.
"""
import os
import sys
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter

from google.colab import userdata



# Sample document with multiple distinct topics
sample_document = """
TechCorp Company Overview

Company History: Founded in 1995 in a garage in Silicon Valley, TechCorp started as a small software consultancy. By 2000, it had grown to 500 employees and went public. The early years were marked by rapid expansion and the release of its flagship product, the TechOS. The founders, Jane Smith and John Doe, built the company on principles of innovation and customer focus.

Product Lineup: Today, TechCorp offers a wide range of enterprise software solutions. The CloudSuite is our most popular offering, providing scalable cloud infrastructure for businesses of all sizes. We also offer DataGuard for enterprise security, protecting sensitive data with military-grade encryption. AI-Core handles machine learning integration, making AI accessible to non-technical teams. Each product is designed to work seamlessly with the others.

Remote Work Policy: Employees may work remotely up to 3 days per week with manager approval. Remote work must be conducted using company-approved devices with VPN access enabled. All employees must be available during core hours (10 AM - 4 PM) and maintain regular communication with their team. Remote work is not a substitute for childcare or eldercare.

Future Vision: Looking ahead, TechCorp is betting big on quantum computing. We plan to invest $1B over the next 5 years in R&D for quantum technologies. Our goal is to be the first company to offer commercial quantum cloud services by 2030. This investment will create new positions for quantum researchers and engineers across all our locations.
"""

print("📄 Sample Document:")
print(f"Length: {len(sample_document)} characters")
print(f"Contains 4 distinct topics: History, Products, Remote Work, Future")
print()

# First, let's compare with basic chunking
print("🔧 Comparison: Basic Chunking vs Agentic Chunking")
print("-" * 50)

# Basic character-based chunking
basic_splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

basic_chunks = basic_splitter.split_text(sample_document)
print(f"\n📊 Basic Chunking Result: {len(basic_chunks)} chunks")
print("   (Based on character count, may split mid-topic)")
for i, chunk in enumerate(basic_chunks, 1):
    preview = chunk[:60].replace('\n', ' ').strip()
    print(f"   Chunk {i}: {preview}...")
print()

# Agentic chunking using LLM
def agentic_chunking(text):
    """
    Uses an LLM to split text into semantically distinct chunks.
    The AI analyzes topic shifts and creates meaningful boundaries.
    """
    print("🤔 Agent is analyzing the document for semantic topic shifts...")

    llm = ChatOpenAI(
        model=MODEL_NAME,
        openai_api_key=API_KEY,
        openai_api_base=API_BASE,
        temperature=0  # Deterministic output for consistency
    )

    # The Prompt: Instruct the LLM to act as a "Chunking Agent"
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert document editor specializing in semantic document analysis.
Your task is to split the provided text into semantically distinct chunks based on topic shifts.

Rules:
1. Keep related sentences together - don't break up a single topic
2. Split ONLY when the topic changes significantly (e.g., History -> Products -> Policy -> Future)
3. Each chunk should be about ONE coherent topic
4. Output the chunks separated by '---SPLIT---'
5. Do not modify the original text - just split it at appropriate boundaries
6. Include section headers with their content in the same chunk"""),
        ("user", "{text}")
    ])

    chain = prompt | llm | StrOutputParser()

    try:
        response = chain.invoke({"text": text})
        # Split the response by our delimiter and clean up
        chunks = [c.strip() for c in response.split("---SPLIT---") if c.strip()]
        return chunks
    except Exception as e:
        print(f"\n❌ API Error: {e}")
        return []

# Run agentic chunking
agentic_chunks = agentic_chunking(sample_document)

if agentic_chunks:
    print(f"\n📊 Agentic Chunking Result: {len(agentic_chunks)} chunks")
    print("   (Based on semantic meaning and topic shifts)")
    print()

    for i, chunk in enumerate(agentic_chunks, 1):
        # Identify the likely topic from the chunk
        if "History" in chunk or "Founded" in chunk:
            topic = "Company History"
        elif "Product" in chunk or "CloudSuite" in chunk:
            topic = "Products"
        elif "Remote" in chunk or "work" in chunk.lower():
            topic = "Remote Work Policy"
        elif "Future" in chunk or "quantum" in chunk.lower():
            topic = "Future Vision"
        else:
            topic = "General"

        print(f"📦 Chunk {i} - Topic: {topic}")
        print(f"   Length: {len(chunk)} characters")
        preview = chunk[:80].replace('\n', ' ').strip()
        print(f"   Preview: {preview}...")
        print()

    # Comparison summary
    print("🔍 Comparison Summary:")
    print("-" * 50)
    print(f"Basic Chunking:   {len(basic_chunks)} chunks (character-based)")
    print(f"Agentic Chunking: {len(agentic_chunks)} chunks (semantic-based)")
    print()
    print("💡 Key Differences:")
    print("✅ Agentic chunking identifies natural topic boundaries")
    print("✅ Each chunk contains ONE coherent topic")
    print("✅ Better semantic coherence for RAG retrieval")
    print("✅ AI understands context and meaning")
    print("✅ No arbitrary character limit splitting")
    print()

    print("💡 When to Use Agentic Chunking:")
    print("✅ Documents with clear topic sections")
    print("✅ When semantic coherence is critical")
    print("✅ Complex documents with mixed content")
    print("✅ When retrieval quality matters more than speed")
    print()

    print("⚠️  Considerations:")
    print("• Requires LLM API calls (cost and latency)")
    print("• Best for smaller documents or preprocessing")
    print("• May need fallback for very large documents")

    # Create completion marker
    with open("agentic_chunking_complete.txt", "w") as f:
        f.write("Agentic chunking demo completed successfully")

    print("\n✅ Agentic chunking demo completed!")
else:
    print("\n⚠️ Agent failed to produce chunks. Check API connection.")


In [ ]:
# 1. Force install the specific compatible versions
!pip install "opentelemetry-api==1.39.1" \
             "opentelemetry-sdk==1.39.1" \
             "opentelemetry-exporter-otlp-proto-common==1.39.1" \
             "opentelemetry-proto==1.39.1" \
             "protobuf>=5.26.1,<6.0.0dev" \
             --force-reinstall

# 2. Restart the runtime automatically
import os
os.kill(os.getpid(), 9)

  Using cached opentelemetry_api-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl.metadata (1.5 kB)
  Using cached opentelemetry_exporter_otlp_proto_common-1.39.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached opentelemetry_proto-1.39.1-py3-none-any.whl.metadata (2.3 kB)
  Using cached importlib_metadata-8.7.1-py3-none-any.whl.metadata (4.7 kB)
  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl.metadata (2.4 kB)
  Using cached zipp-3.23.0-py3-none-any.whl.metadata (3.6 kB)
Using cached opentelemetry_api-1.39.1-py3-none-any.whl (66 kB)
Using cached opentelemetry_sdk-1.39.1-py3-none-any.whl (132 kB)
Using cached opentelemetry_exporter_otlp_proto_common-1.39.1-py3-none-any.whl (18 kB)
Using cached opentelemetry_proto-1.39.1-py3-none-any.whl (72 kB)
Using cached opentelemetry_semantic_conventions-0.60b1-py3-none-any.whl (219 kB)
   ━━━━━━━━━

In [1]:
!pip install -q  chromadb sentence-transformers langchain-text-splitters


In [2]:
#!/usr/bin/env python3
"""
Chunked Vector Search Demo
Compare search performance with and without chunking
"""

import chromadb
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("🔍 Chunked Vector Search Demo")
print("=" * 50)

# Initialize ChromaDB and model
client = chromadb.Client()
model = SentenceTransformer('all-MiniLM-L6-v2')

# Sample policy document
policy_document = """
TechCorp Remote Work Policy

Section 1: Eligibility and Approval
Employees may work remotely up to 3 days per week with manager approval.
Remote work days must be scheduled in advance and approved by your direct supervisor.
All remote work must comply with company security policies and use approved equipment.

Section 2: Equipment Requirements
Remote employees must have a secure and reliable internet connection with minimum speeds of 25 Mbps download and 5 Mbps upload.
All work must be performed on company-approved devices and software.
Employees must use VPN when accessing company systems.
Personal devices are not permitted for work purposes.

Section 3: Workspace Standards
Remote work is not a substitute for childcare or eldercare responsibilities.
Employees must have a dedicated workspace free from distractions.
The workspace must be professional and suitable for video calls.
Background noise should be minimized during meetings.

Section 4: Communication Requirements
Employees must be available during core business hours (9 AM - 5 PM local time).
Regular check-ins with managers are required.
Team meetings must be attended via video conference.
Email and instant messaging should be checked regularly.

Section 5: Security and Compliance
All company data must be handled according to security policies.
Confidential information must not be discussed in public spaces.
Documents must be stored in approved cloud systems only.
Regular security training must be completed.
"""

print("📄 Sample Policy Document:")
print(f"Length: {len(policy_document)} characters")
print()

# Test 1: Search WITHOUT chunking
print("🔧 Test 1: Search WITHOUT Chunking")
print("-" * 40)

# Create collection for non-chunked search
collection_no_chunking = client.create_collection("no_chunking")

# Store entire document as single chunk
collection_no_chunking.add(
    documents=[policy_document],
    ids=["full_document"]
)

print("Stored entire document as single chunk")
print()

# Test 2: Search WITH chunking
print("🔧 Test 2: Search WITH Chunking")
print("-" * 40)

# Create collection for chunked search
collection_chunked = client.create_collection("chunked")

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = splitter.split_text(policy_document)
print(f"Split document into {len(chunks)} chunks")

# Store chunks in vector database
chunk_ids = [f"chunk_{i+1}" for i in range(len(chunks))]
collection_chunked.add(
    documents=chunks,
    ids=chunk_ids
)

print("Stored chunks in vector database")
print()

# Test queries
test_queries = [
    "What are the internet speed requirements?",
    "Can I use my personal laptop for work?",
    "What are the workspace requirements?",
    "How often do I need to check in with my manager?"
]

print("🔍 Search Performance Comparison:")
print("=" * 50)

for query in test_queries:
    print(f"\nQuery: '{query}'")
    print("-" * 30)

    # Search without chunking
    results_no_chunking = collection_no_chunking.query(
        query_texts=[query],
        n_results=1
    )

    # Search with chunking
    results_chunked = collection_chunked.query(
        query_texts=[query],
        n_results=2
    )

    print("Without Chunking:")
    print(f"  Similarity: {1 - results_no_chunking['distances'][0][0]:.3f}")
    print(f"  Result: {results_no_chunking['documents'][0][0][:100]}...")
    print(f"  Problem: Returns entire document!")

    print("\nWith Chunking:")
    for i, (doc, distance) in enumerate(zip(results_chunked['documents'][0], results_chunked['distances'][0])):
        similarity = 1 - distance
        print(f"  Chunk {i+1} - Similarity: {similarity:.3f}")
        print(f"  Result: {doc[:100]}...")
        print(f"  Benefit: Focused, relevant information!")

print("\n💡 Chunking Benefits for Search:")
print("✅ More precise and relevant results")
print("✅ Focused information instead of entire documents")
print("✅ Better similarity scores for specific topics")
print("✅ Easier to find specific information")
print("✅ Improved user experience")
print("✅ Better context for LLM generation")

print("\n📊 Performance Summary:")
print(f"Without chunking: 1 large document, hard to find specific info")
print(f"With chunking: {len(chunks)} focused chunks, precise results")

# Create completion marker
with open("chunked_search_complete.txt", "w") as f:
    f.write("Chunked search demo completed successfully")

print("\n✅ Chunked search demo completed!")


🔍 Chunked Vector Search Demo


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📄 Sample Policy Document:
Length: 1492 characters

🔧 Test 1: Search WITHOUT Chunking
----------------------------------------


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:08<00:00, 9.64MiB/s]


Stored entire document as single chunk

🔧 Test 2: Search WITH Chunking
----------------------------------------
Split document into 7 chunks
Stored chunks in vector database

🔍 Search Performance Comparison:

Query: 'What are the internet speed requirements?'
------------------------------
Without Chunking:
  Similarity: -0.449
  Result: 
TechCorp Remote Work Policy

Section 1: Eligibility and Approval
Employees may work remotely up to ...
  Problem: Returns entire document!

With Chunking:
  Chunk 1 - Similarity: 0.064
  Result: Section 2: Equipment Requirements
Remote employees must have a secure and reliable internet connecti...
  Benefit: Focused, relevant information!
  Chunk 2 - Similarity: -0.452
  Result: Section 4: Communication Requirements
Employees must be available during core business hours (9 AM -...
  Benefit: Focused, relevant information!

Query: 'Can I use my personal laptop for work?'
------------------------------
Without Chunking:
  Similarity: -0.283
  Result: 
T